# Deteccion de caja azul en HSI

Este notebook abre imagenes hiperespectrales ENVI (`.hdr`), genera una pseudo-RGB y detecta la caja azul donde se encuentra el especimen.

La funcion principal es `open_hsi_and_detect_blue_box(...)`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import rgb_to_hsv
import numpy as np
from scipy import ndimage as ndi

plt.rcParams['figure.dpi'] = 120


In [ ]:
SPECIMENS = [
    {
        'id': 'SB012',
        'hsi_hdr_path': Path(r'Datos\SB012\SB012\SB012 HSI\SB012_M1_002\SB012_M1_nrm_edu.hdr'),
    },
    {
        'id': 'SB013',
        'hsi_hdr_path': Path(r'Datos\SB013\SB013\SB013_001\SB013_nrm_edu.hdr'),
    },
    {
        'id': 'SB017',
        'hsi_hdr_path': Path(r'Datos\SB017_uomo\SB017_uomo\SB017_001\SB017_nrm_edu.hdr'),
    },
    {
        'id': 'SB018',
        'hsi_hdr_path': Path(r'Datos\SB018\SB018\SB018_001\SB018_nrm_edu.hdr'),
    },
    {
        'id': 'SB019',
        'hsi_hdr_path': Path(r'Datos\SB019\SB019\SB019_001\SB019_nrm_edu.hdr'),
    },
    {
        'id': 'SB020',
        'hsi_hdr_path': Path(r'Datos\SB020\SB020\SB020_001\SB020_nrm_edu.hdr'),
    },
]

for specimen in SPECIMENS:
    print(specimen['id'], '| HSI existe:', specimen['hsi_hdr_path'].exists(), '|', specimen['hsi_hdr_path'])


In [ ]:
def parse_envi_header(hdr_path):
    hdr_path = Path(hdr_path)
    text = hdr_path.read_text(encoding='utf-8', errors='ignore')
    metadata = {}
    current_key = None
    current_value = []

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.upper() == 'ENVI':
            continue

        if current_key is not None:
            current_value.append(line)
            if '}' in line:
                metadata[current_key] = ' '.join(current_value)
                current_key = None
                current_value = []
            continue

        if '=' not in line:
            continue

        key, value = line.split('=', 1)
        key = key.strip().lower()
        value = value.strip()

        if value.startswith('{') and '}' not in value:
            current_key = key
            current_value = [value]
        else:
            metadata[key] = value

    return metadata


def parse_wavelengths(metadata):
    values = metadata.get('wavelength')
    if values is None:
        return None

    values = values.strip().strip('{}')
    return np.array([float(v.strip()) for v in values.split(',') if v.strip()], dtype=np.float32)


def envi_dtype(metadata):
    data_type = int(metadata['data type'])
    byte_order = int(metadata.get('byte order', 0))
    endian = '<' if byte_order == 0 else '>'
    dtype_map = {
        1: 'u1',
        2: 'i2',
        3: 'i4',
        4: 'f4',
        5: 'f8',
        12: 'u2',
        13: 'u4',
        14: 'i8',
        15: 'u8',
    }
    if data_type not in dtype_map:
        raise ValueError(f'Tipo de dato ENVI no soportado: {data_type}')
    return np.dtype(endian + dtype_map[data_type])


def find_envi_data_path(hdr_path, metadata):
    hdr_path = Path(hdr_path)
    candidates = []

    for key in ('data file', 'file name', 'image'):
        if key in metadata:
            value = metadata[key].strip().strip('{}').strip('"')
            candidates.append(hdr_path.parent / value)

    candidates.extend([
        hdr_path.with_suffix(''),
        hdr_path.parent / hdr_path.stem.replace('_edu', ''),
    ])

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f'No se encontro el binario ENVI asociado a {hdr_path.name}')


def read_envi_band(hdr_path, metadata, band_idx):
    data_path = find_envi_data_path(hdr_path, metadata)
    samples = int(metadata['samples'])
    lines = int(metadata['lines'])
    bands = int(metadata['bands'])
    offset = int(metadata.get('header offset', 0))
    interleave = metadata.get('interleave', 'bsq').strip().lower()
    dtype = envi_dtype(metadata)

    if interleave == 'bsq':
        cube = np.memmap(data_path, dtype=dtype, mode='r', offset=offset, shape=(bands, lines, samples))
        return np.asarray(cube[band_idx], dtype=np.float32)
    if interleave == 'bil':
        cube = np.memmap(data_path, dtype=dtype, mode='r', offset=offset, shape=(lines, bands, samples))
        return np.asarray(cube[:, band_idx, :], dtype=np.float32)
    if interleave == 'bip':
        cube = np.memmap(data_path, dtype=dtype, mode='r', offset=offset, shape=(lines, samples, bands))
        return np.asarray(cube[:, :, band_idx], dtype=np.float32)

    raise ValueError(f'Interleave ENVI no soportado: {interleave}')


def robust_normalize(channel, p_low=2, p_high=98):
    channel = np.asarray(channel, dtype=np.float32)
    if not np.any(np.isfinite(channel)):
        return np.zeros_like(channel, dtype=np.float32)

    lo = np.nanpercentile(channel, p_low)
    hi = np.nanpercentile(channel, p_high)

    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(channel, dtype=np.float32)

    normalized = np.clip((channel - lo) / (hi - lo), 0, 1)
    return np.nan_to_num(normalized, nan=0.0, posinf=1.0, neginf=0.0).astype(np.float32)


def hsi_pseudo_rgb_from_path(hdr_path, r_nm=650, g_nm=550, b_nm=450):
    hdr_path = Path(hdr_path)
    if not hdr_path.exists():
        raise FileNotFoundError(f'No existe la HSI: {hdr_path}')

    metadata = parse_envi_header(hdr_path)
    wavelengths = parse_wavelengths(metadata)

    if wavelengths is None:
        band_count = int(metadata['bands'])
        r_idx = min(band_count - 1, int(round(0.72 * (band_count - 1))))
        g_idx = min(band_count - 1, int(round(0.50 * (band_count - 1))))
        b_idx = min(band_count - 1, int(round(0.28 * (band_count - 1))))
        band_info = {'r_idx': r_idx, 'g_idx': g_idx, 'b_idx': b_idx}
    else:
        r_idx = int(np.argmin(np.abs(wavelengths - r_nm)))
        g_idx = int(np.argmin(np.abs(wavelengths - g_nm)))
        b_idx = int(np.argmin(np.abs(wavelengths - b_nm)))
        band_info = {
            'r_idx': r_idx,
            'r_nm': float(wavelengths[r_idx]),
            'g_idx': g_idx,
            'g_nm': float(wavelengths[g_idx]),
            'b_idx': b_idx,
            'b_nm': float(wavelengths[b_idx]),
        }

    rgb = np.stack(
        [
            robust_normalize(read_envi_band(hdr_path, metadata, r_idx)),
            robust_normalize(read_envi_band(hdr_path, metadata, g_idx)),
            robust_normalize(read_envi_band(hdr_path, metadata, b_idx)),
        ],
        axis=-1,
    )
    rgb = np.nan_to_num(rgb, nan=0.0, posinf=1.0, neginf=0.0)
    rgb = (np.clip(rgb, 0, 1) * 255).astype(np.uint8)

    info = {
        'shape': rgb.shape,
        'data_path': find_envi_data_path(hdr_path, metadata),
        **band_info,
    }
    return rgb, info


In [ ]:
def draw_rectangle_rgb(img_rgb, bbox, color=(255, 255, 0), thickness=4):
    out = img_rgb.copy()
    x1, y1, x2, y2 = bbox
    h, w = out.shape[:2]
    x1 = int(np.clip(x1, 0, w - 1))
    x2 = int(np.clip(x2, 0, w - 1))
    y1 = int(np.clip(y1, 0, h - 1))
    y2 = int(np.clip(y2, 0, h - 1))
    t = max(1, int(thickness))

    out[y1:y1 + t, x1:x2 + 1] = color
    out[max(y2 - t + 1, 0):y2 + 1, x1:x2 + 1] = color
    out[y1:y2 + 1, x1:x1 + t] = color
    out[y1:y2 + 1, max(x2 - t + 1, 0):x2 + 1] = color
    return out


def overlay_mask_rgb(img_rgb, mask, color=(0, 255, 255), alpha=0.35):
    out = img_rgb.copy().astype(np.float32)
    color_arr = np.array(color, dtype=np.float32)
    mask = mask.astype(bool)
    out[mask] = (1 - alpha) * out[mask] + alpha * color_arr
    return np.clip(out, 0, 255).astype(np.uint8)


def detect_blue_box_from_rgb(
    rgb_u8,
    hue_min=0.48,
    hue_max=0.66,
    sat_min=0.10,
    value_min=0.10,
    value_max=0.90,
    blue_red_min=0.090,
    green_red_min=-0.005,
    blue_excess_min=0.075,
    white_sat_max=0.080,
    white_value_min=0.82,
    open_size=3,
    close_size=25,
    dilate_size=7,
    min_area=1200,
    padding=20,
):
    """Detecta la caja azul de la pseudo-RGB y devuelve mascara, bbox y overlay."""
    rgb_u8 = np.asarray(rgb_u8, dtype=np.uint8)
    rgb = rgb_u8.astype(np.float32) / 255.0
    hsv = rgb_to_hsv(rgb)

    h = hsv[:, :, 0]
    s = hsv[:, :, 1]
    v = hsv[:, :, 2]
    r = rgb[:, :, 0]
    g = rgb[:, :, 1]
    b = rgb[:, :, 2]

    blue_hue = (h >= hue_min) & (h <= hue_max)
    blue_red = b - r
    green_red = g - r
    blue_excess = 0.65 * blue_red + 0.35 * green_red
    blue_dominance = (blue_red > blue_red_min) & (green_red > green_red_min) & (blue_excess > blue_excess_min)
    not_extreme = (v >= value_min) & (v <= value_max)
    not_white = ~((v > white_value_min) & (s < white_sat_max))
    mask = blue_hue & blue_dominance & (s >= sat_min) & not_extreme & not_white

    if open_size > 1:
        mask = ndi.binary_opening(mask, structure=np.ones((open_size, open_size), dtype=bool))
    if close_size > 1:
        mask = ndi.binary_closing(mask, structure=np.ones((close_size, close_size), dtype=bool))
    if dilate_size > 1:
        mask = ndi.binary_dilation(mask, structure=np.ones((dilate_size, dilate_size), dtype=bool))

    labels, num_labels = ndi.label(mask)
    if num_labels == 0:
        raise ValueError('No se detecto ninguna region azul. Ajusta los umbrales HSV/dominancia.')

    areas = np.bincount(labels.ravel())
    areas[0] = 0
    kept_labels = np.where(areas >= min_area)[0]
    if len(kept_labels) == 0:
        kept_labels = np.array([int(np.argmax(areas))])

    clean_mask = np.isin(labels, kept_labels)
    clean_mask = ndi.binary_fill_holes(clean_mask)

    ys, xs = np.where(clean_mask)
    if len(xs) == 0:
        raise ValueError('La mascara azul quedo vacia despues de limpiar componentes.')

    img_h, img_w = clean_mask.shape
    x1 = max(int(xs.min()) - padding, 0)
    y1 = max(int(ys.min()) - padding, 0)
    x2 = min(int(xs.max()) + padding, img_w - 1)
    y2 = min(int(ys.max()) + padding, img_h - 1)
    bbox = (x1, y1, x2, y2)

    overlay = overlay_mask_rgb(rgb_u8, clean_mask, color=(0, 220, 255), alpha=0.30)
    overlay = draw_rectangle_rgb(overlay, bbox, color=(255, 255, 0), thickness=4)

    info = {
        'bbox_xyxy': bbox,
        'bbox_width': x2 - x1 + 1,
        'bbox_height': y2 - y1 + 1,
        'mask_area_px': int(clean_mask.sum()),
        'kept_components': int(len(kept_labels)),
        'sat_min': sat_min,
        'blue_red_min': blue_red_min,
        'blue_excess_min': blue_excess_min,
    }
    return clean_mask, bbox, overlay, info


def open_hsi_and_detect_blue_box(hdr_path, specimen_id=None, show=True, **detect_kwargs):
    """Abre una HSI, genera pseudo-RGB y detecta la caja azul."""
    rgb_u8, hsi_info = hsi_pseudo_rgb_from_path(hdr_path)
    mask, bbox, overlay, box_info = detect_blue_box_from_rgb(rgb_u8, **detect_kwargs)

    result = {
        'specimen_id': specimen_id,
        'hdr_path': Path(hdr_path),
        'pseudo_rgb': rgb_u8,
        'blue_mask': mask,
        'overlay': overlay,
        'bbox_xyxy': bbox,
        'hsi_info': hsi_info,
        'box_info': box_info,
    }

    if show:
        plot_blue_box_detection(result)

    return result


def plot_blue_box_detection(result):
    specimen_id = result['specimen_id'] or result['hdr_path'].stem
    hsi_info = result['hsi_info']
    box_info = result['box_info']
    band_text = f"R={hsi_info.get('r_nm', hsi_info['r_idx']):.1f}, G={hsi_info.get('g_nm', hsi_info['g_idx']):.1f}, B={hsi_info.get('b_nm', hsi_info['b_idx']):.1f}"

    fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

    axes[0].imshow(result['pseudo_rgb'])
    axes[0].set_title(f'{specimen_id} - HSI pseudo-RGB\n{band_text}')

    axes[1].imshow(result['blue_mask'], cmap='gray')
    axes[1].set_title('Mascara caja azul')

    axes[2].imshow(result['overlay'])
    axes[2].set_title(
        'Caja azul detectada\n'
        f"bbox={box_info['bbox_xyxy']} | area={box_info['mask_area_px']} px"
    )

    for ax in axes:
        ax.axis('off')

    plt.show()


In [ ]:
results = {}

for specimen in SPECIMENS:
    print(f"\n=== {specimen['id']} ===")
    result = open_hsi_and_detect_blue_box(
        specimen['hsi_hdr_path'],
        specimen_id=specimen['id'],
        show=True,
    )
    results[specimen['id']] = result
    print(result['box_info'])


## Uso con una sola imagen

Para usar la funcion con una HSI concreta:

```python
result = open_hsi_and_detect_blue_box(r'RUTA_A_LA_IMAGEN.hdr', specimen_id='SBXXX')
bbox = result['bbox_xyxy']
mask = result['blue_mask']
```

## Ajuste fino del contorno detectado

Estas funciones no cambian la deteccion original. Toman un `result` ya calculado y devuelven una copia con la mascara y la caja ligeramente reducidas.


In [ ]:
def shrink_blue_box_detection(
    result,
    shrink_pixels=10,
    bbox_padding=0,
    kernel_shape='ellipse',
    show=True,
):
    """
    Reduce ligeramente el contorno de la caja azul ya detectada.

    No modifica el result original. Devuelve una copia con:
        - blue_mask_shrunk
        - overlay_shrunk
        - bbox_xyxy_shrunk
        - box_info_shrunk

    Parametros
    ----------
    result : dict
        Salida de open_hsi_and_detect_blue_box(...).
    shrink_pixels : int
        Pixeles aproximados que se erosiona la mascara azul hacia dentro.
    bbox_padding : int
        Margen extra que se vuelve a sumar al bbox despues de erosionar.
        Usa 0 para el ajuste mas estricto, o 5-15 si queda demasiado cerrado.
    kernel_shape : {'ellipse', 'rect'}
        Forma del kernel de erosion. 'ellipse' suele conservar mejor esquinas suaves.
    show : bool
        Si True, muestra comparacion entre deteccion original y reducida.
    """
    rgb_u8 = np.asarray(result['pseudo_rgb'], dtype=np.uint8)
    original_mask = np.asarray(result['blue_mask']).astype(bool)
    shrink_pixels = int(max(0, shrink_pixels))
    bbox_padding = int(max(0, bbox_padding))

    if shrink_pixels == 0:
        shrunk_mask = original_mask.copy()
    else:
        try:
            import cv2

            mask_u8 = original_mask.astype(np.uint8) * 255
            k = 2 * shrink_pixels + 1
            if kernel_shape == 'rect':
                kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (k, k))
            else:
                kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
            shrunk_mask = cv2.erode(mask_u8, kernel, iterations=1) > 0
        except Exception:
            structure = np.ones((2 * shrink_pixels + 1, 2 * shrink_pixels + 1), dtype=bool)
            shrunk_mask = ndi.binary_erosion(original_mask, structure=structure)

    if not np.any(shrunk_mask):
        raise ValueError('La erosion ha vaciado la mascara. Prueba con menos shrink_pixels.')

    ys, xs = np.where(shrunk_mask)
    img_h, img_w = shrunk_mask.shape
    x1 = max(int(xs.min()) - bbox_padding, 0)
    y1 = max(int(ys.min()) - bbox_padding, 0)
    x2 = min(int(xs.max()) + bbox_padding, img_w - 1)
    y2 = min(int(ys.max()) + bbox_padding, img_h - 1)
    bbox = (x1, y1, x2, y2)

    overlay = overlay_mask_rgb(rgb_u8, shrunk_mask, color=(255, 180, 0), alpha=0.35)
    overlay = draw_rectangle_rgb(overlay, bbox, color=(255, 255, 0), thickness=4)

    adjusted = result.copy()
    adjusted['blue_mask_shrunk'] = shrunk_mask
    adjusted['overlay_shrunk'] = overlay
    adjusted['bbox_xyxy_shrunk'] = bbox
    adjusted['box_info_shrunk'] = {
        'bbox_xyxy': bbox,
        'bbox_width': x2 - x1 + 1,
        'bbox_height': y2 - y1 + 1,
        'mask_area_px': int(shrunk_mask.sum()),
        'shrink_pixels': shrink_pixels,
        'bbox_padding': bbox_padding,
        'kernel_shape': kernel_shape,
    }

    if show:
        plot_shrunk_blue_box_detection(adjusted)

    return adjusted


def plot_shrunk_blue_box_detection(adjusted_result):
    specimen_id = adjusted_result.get('specimen_id') or adjusted_result['hdr_path'].stem
    original_info = adjusted_result['box_info']
    shrunk_info = adjusted_result['box_info_shrunk']

    fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

    axes[0].imshow(adjusted_result['overlay'])
    axes[0].set_title(
        'Deteccion original\n'
        f"bbox={original_info['bbox_xyxy']}"
    )

    axes[1].imshow(adjusted_result['blue_mask_shrunk'], cmap='gray')
    axes[1].set_title(
        f"Mascara reducida | shrink={shrunk_info['shrink_pixels']} px\n"
        f"area={shrunk_info['mask_area_px']} px"
    )

    axes[2].imshow(adjusted_result['overlay_shrunk'])
    axes[2].set_title(
        f'{specimen_id} - caja ajustada\n'
        f"bbox={shrunk_info['bbox_xyxy']}"
    )

    for ax in axes:
        ax.axis('off')

    plt.show()


def shrink_all_blue_box_detections(results, shrink_pixels=10, bbox_padding=0, show=True):
    """Aplica shrink_blue_box_detection a todos los results ya calculados."""
    adjusted_results = {}
    for specimen_id, result in results.items():
        print(f"\n=== {specimen_id} | shrink={shrink_pixels}px ===")
        adjusted = shrink_blue_box_detection(
            result,
            shrink_pixels=shrink_pixels,
            bbox_padding=bbox_padding,
            show=show,
        )
        adjusted_results[specimen_id] = adjusted
        print(adjusted['box_info_shrunk'])
    return adjusted_results



In [ ]:
adjusted_results = shrink_all_blue_box_detections(
    results,
    shrink_pixels=10,
    bbox_padding=0,
    show=True,
)